In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Volumes/workspace/default/worldcup_2026"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

fifa_rankings_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/fifa_rankings.csv")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

fifa_rankings_raw.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.bronze.fifa_rankings_raw")

print("Ranking FIFA cargado en Bronze.")

In [0]:
display(spark.table("workspace.bronze.fifa_rankings_raw").limit(10))
spark.table("workspace.bronze.fifa_rankings_raw").printSchema()

In [0]:
from pyspark.sql.functions import col, to_date, expr

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

fifa_raw = spark.table("workspace.bronze.fifa_rankings_raw")

# Eliminar columnas innecesarias si existen
columns_to_drop = [c for c in fifa_raw.columns if c.lower().startswith("unnamed")]
fifa_raw = fifa_raw.drop(*columns_to_drop)

fifa_clean = (
    fifa_raw
    .select(
        expr("try_cast(rank as int)").alias("fifa_rank"),
        col("country_full").alias("country_full"),
        col("country_abrv").alias("country_abrv"),
        expr("try_cast(total_points as double)").alias("fifa_points"),
        expr("try_cast(previous_points as double)").alias("previous_points"),
        expr("try_cast(rank_change as int)").alias("rank_change"),
        col("confederation").alias("confederation"),
        to_date(col("rank_date")).alias("fifa_rank_date")
    )
    .dropDuplicates()
)

fifa_clean.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.silver.fifa_rankings_clean")

print("Ranking FIFA limpio guardado en Silver.")

In [0]:
display(spark.table("workspace.silver.fifa_rankings_clean").limit(20))
print("Filas FIFA:", spark.table("workspace.silver.fifa_rankings_clean").count())

In [0]:
import pandas as pd
import numpy as np
import unicodedata
import re

matches = spark.table("workspace.gold.training_dataset_features").toPandas()
fifa = spark.table("workspace.silver.fifa_rankings_clean").toPandas()

matches["date"] = pd.to_datetime(matches["date"])
fifa["fifa_rank_date"] = pd.to_datetime(fifa["fifa_rank_date"])

matches = matches.sort_values("date").reset_index(drop=True)
matches["match_id"] = matches.index

def name_key(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.replace("&", " and ")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    mapping = {
        "ivory coast": "cote divoire",
        "cote d ivoire": "cote divoire",
        "cote divoire": "cote divoire",
        "dr congo": "congo dr",
        "democratic republic of congo": "congo dr",
        "south korea": "korea republic",
        "north korea": "korea dpr",
        "czech republic": "czechia",
        "usa": "united states",
        "united states of america": "united states",
        "turkey": "turkiye",
        "curacao": "curacao",
        "cape verde": "cabo verde",
        "swaziland": "eswatini",
        "macedonia": "north macedonia",
        "iran": "ir iran",
        "syria": "syria",
        "russia": "russia",
        "kyrgyzstan": "kyrgyz republic",
        "brunei": "brunei darussalam",
        "east timor": "timor leste",
        "laos": "laos",
        "moldova": "moldova",
        "vietnam": "vietnam",
        "bosnia and herzegovina": "bosnia and herzegovina"
    }
    
    return mapping.get(name, name)

fifa["country_key"] = fifa["country_full"].apply(name_key)

In [0]:
def add_fifa_ranking(matches_df, fifa_df, side):
    team_col = f"{side}_team"
    
    base = matches_df[["match_id", "date", team_col]].copy()
    base["team_key"] = base[team_col].apply(name_key)
    
    output_parts = []
    
    for team_key, team_matches in base.groupby("team_key"):
        team_matches = team_matches.sort_values("date")
        
        team_rankings = (
            fifa_df[fifa_df["country_key"] == team_key]
            .sort_values("fifa_rank_date")
        )
        
        if team_rankings.empty:
            temp = team_matches.copy()
            temp[f"fifa_rank_{side}"] = np.nan
            temp[f"fifa_points_{side}"] = np.nan
            temp[f"confederation_{side}"] = None
            temp[f"fifa_rank_date_{side}"] = pd.NaT
            output_parts.append(temp[[
                "match_id",
                f"fifa_rank_{side}",
                f"fifa_points_{side}",
                f"confederation_{side}",
                f"fifa_rank_date_{side}"
            ]])
            continue
        
        merged = pd.merge_asof(
            team_matches,
            team_rankings[[
                "fifa_rank_date",
                "fifa_rank",
                "fifa_points",
                "confederation",
                "country_key"
            ]],
            left_on="date",
            right_on="fifa_rank_date",
            direction="backward"
        )
        
        merged = merged.rename(columns={
            "fifa_rank": f"fifa_rank_{side}",
            "fifa_points": f"fifa_points_{side}",
            "confederation": f"confederation_{side}",
            "fifa_rank_date": f"fifa_rank_date_{side}"
        })
        
        output_parts.append(merged[[
            "match_id",
            f"fifa_rank_{side}",
            f"fifa_points_{side}",
            f"confederation_{side}",
            f"fifa_rank_date_{side}"
        ]])
    
    return pd.concat(output_parts, ignore_index=True)

In [0]:
home_fifa = add_fifa_ranking(matches, fifa, "home")
away_fifa = add_fifa_ranking(matches, fifa, "away")

matches_fifa = (
    matches
    .merge(home_fifa, on="match_id", how="left")
    .merge(away_fifa, on="match_id", how="left")
)

matches_fifa["fifa_rank_diff"] = matches_fifa["fifa_rank_away"] - matches_fifa["fifa_rank_home"]
matches_fifa["fifa_points_diff"] = matches_fifa["fifa_points_home"] - matches_fifa["fifa_points_away"]

display(matches_fifa[[
    "date",
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff",
    "result"
]].head(30))

In [0]:
modern = matches_fifa[matches_fifa["year"] >= 2000].copy()

missing_home = (
    modern[modern["fifa_rank_home"].isna()]
    ["home_team"]
    .value_counts()
    .head(30)
)

missing_away = (
    modern[modern["fifa_rank_away"].isna()]
    ["away_team"]
    .value_counts()
    .head(30)
)

print("Faltantes home desde 2000:")
display(missing_home.reset_index())

print("Faltantes away desde 2000:")
display(missing_away.reset_index())

In [0]:
matches_fifa_spark = spark.createDataFrame(matches_fifa)

matches_fifa_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.training_dataset_features_fifa")

print("Tabla Gold con ranking FIFA creada correctamente.")

In [0]:
display(spark.table("workspace.gold.training_dataset_features_fifa").limit(10))

print("Filas:", spark.table("workspace.gold.training_dataset_features_fifa").count())